# matcher Tutorial: Entity Resolution and Deduplication

Welcome. This tutorial is a hands-on path from "I have two tables" (or one table for dedup) to a pipeline you can tune and trust: exact rules, fuzzy matching, optional blocking, and a single blended score. We use small sample datasets so runs are fast and you can reproduce every step—but the ideas scale.

**What you'll get:** You'll *design* matching strategies, *run* them, and *evaluate* the results. The goal isn't to click through—it's to build the habit of measuring, comparing, and deciding based on your data. Matcher is built for that loop.

---

## Prerequisites

- **Python 3.10+** and **uv** (or pip). From the repository root: `uv sync` (or `pip install -e .`) so matcher is installed.
- **To run notebooks:** Jupyter is not in the main dependency set. Use a venv with the tutorial deps: from repo root, `uv venv .venv-tutorial`, then `source .venv-tutorial/bin/activate` (Windows: `.venv-tutorial\Scripts\activate`), then `uv sync --group tutorial`. See [README](README.md) for details.
- **Where to run:** Open and run notebooks from the **repository root** or from `docs/tutorial`. Sample data is **generated on the fly** by the `tutorial_data` package—no need to run a data-generation script or commit data files.

## Table of Contents

Work through in order; each link opens the notebook in VS Code, Cursor, or your Jupyter environment. We introduce the **measurement loop** early (after exact matching), then add **fuzzy** and **blocking** as building blocks, then **design** (combos, blended score), then **deduplication**.

| # | Notebook | What you'll do |
|---|----------|----------------|
| 1 | [**01 Preparation**](01_preparation.ipynb) | Get your data match-ready: schema, nulls, standardization, derived columns, and ground truth. |
| 2 | [**02 Exact Matching**](02_exact_matching.ipynb) | Single rule, cascading rules, and the evaluate API—the first building block. |
| 3 | [**03 The Measurement Loop**](03_measurement_loop.ipynb) | Measure → change one thing → measure again → compare. The habit for every decision. |
| 4 | [**04 Fuzzy Matching**](04_fuzzy_matching.ipynb) | Similarity on one string field, confidence scores, and threshold tuning. |
| 5 | [**05 Design Your Algorithm**](05_design_algorithm.ipynb) | Exact+fuzzy combo, fuzzy first + exact last, blended score. |
| 6 | [**06 Blocking**](06_blocking.ipynb) | Restrict comparisons to a key (e.g. zip); when it helps and when it hurts recall. |
| 7 | [**07 Deduplication**](07_deduplication.ipynb) | Same ideas on a single table with `Deduplicator`. |

**Suggested path:** [01](01_preparation.ipynb) → [02](02_exact_matching.ipynb) → [03](03_measurement_loop.ipynb) → [04](04_fuzzy_matching.ipynb) → [05](05_design_algorithm.ipynb). Then [06](06_blocking.ipynb) and [07](07_deduplication.ipynb) when you're ready.

## Sample Datasets Used

Data is **generated on the fly** by the `tutorial_data` package (no need to commit or generate files). From repo root, add `docs/tutorial` to your path and import loaders:

```python
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve() / "docs" / "tutorial"))
from tutorial_data import (
    load_evaluation,
    load_fuzzy_evaluation,
    load_blocking_evaluation,
    load_entity_resolution,
    load_entity_resolution_standardized,
    load_deduplication,
)
```

| Loader | Purpose |
|--------|--------|
| `load_entity_resolution()` | Raw data: 500×500, 30 pairs (01—clean it). |
| `load_entity_resolution_standardized()` | Same data, cleaned (02–06: matching, design, blocking; includes exact-name, email-adds-value, and fuzzy-name pairs). |
| `load_evaluation()` | 50×50, 30 perfect pairs; optional alternate for smaller runs. |
| `load_fuzzy_evaluation()` | 50×50, 30 pairs (15 identical, 15 name variants); optional alternate for fuzzy demos (notebooks use ER data). |
| `load_blocking_evaluation()` | 50×50, 30 pairs (15 same zip, 15 split zip); optional alternate for 06 (notebook uses ER data). |
| `load_deduplication()` | Single table, 1000 rows, 50 duplicate pairs (07). |

Optional: run `uv run python scripts/generate_test_data.py` from repo root to write the same data to `data/` for inspection or tests.

## Quick sanity check

Run the cell below to confirm your environment and the tutorial_data package. Data is loaded on the fly—no need to generate files first. If it runs without errors, you're set to start [01 Preparation](01_preparation.ipynb).

In [2]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

# Import tutorial data (run notebooks from repo root, or from docs/tutorial)
_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution_standardized

left, right, _ = load_entity_resolution_standardized()
print(f"Left: {left.shape[0]} rows, Right: {right.shape[0]} rows")
print(f"Columns: {left.columns}")
matcher = Matcher(left=left, right=right, left_id="id", right_id="id")
results = matcher.match(rules="email_clean")
print(f"Matches on email_clean: {results.count}")
print()
print(
    "✓ Environment OK — tutorial_data and matcher are ready. "
    "Start with [01 Preparation](01_preparation.ipynb)."
)

Left: 500 rows, Right: 500 rows
Columns: ['id', 'email', 'first_name', 'last_name', 'address', 'city', 'state', 'zip_code', 'email_clean', 'full_name']
Matches on email_clean: 10

✓ Environment OK — tutorial_data and matcher are ready. Start with [01 Preparation](01_preparation.ipynb).
